# Direct Search Methods for Optimization



Unlike gradient-based algorithms, direct search methods do not require the gradient or Hessian of the objective function. They are useful when:

- derivatives are unavailable;
- derivatives are expensive to compute;
- the objective is noisy or non-smooth;
- the function is available only through evaluations.

We will study:

- cyclic coordinate search;
- cyclic coordinate search with an acceleration step;
- Powell's method;
- the Hooke–Jeeves pattern search method.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the idea of derivative-free direct search.
2. Use one-dimensional minimization inside a multidimensional search method.
3. Implement and interpret cyclic coordinate search.
4. Explain the purpose of an acceleration direction.
5. Describe the main idea behind Powell's method.
6. Apply Hooke–Jeeves search.
7. Compare the behavior of several direct-search algorithms.


In [ ]:
using Logging

global_logger(ConsoleLogger(stderr, Logging.Info; show_limited=false))

In [ ]:
using Plots, Optim, LinearAlgebra

## 1. Test function: Wheeler's ridge

We use **Wheeler's ridge** as the main test problem.

It is a two-dimensional function with a single global minimum. Because the valley is curved, it is useful for illustrating the behavior of search methods that move along selected directions.

The function used in this notebook is

$$
f(x_1,x_2)
=
-\exp\left(
-(x_1x_2-a)^2-(x_2-a)^2
\right),
$$

where the default parameter is $a=1.5$.

Before running any optimization algorithm, it is useful to visualize the objective function and understand the shape of the landscape.


In [ ]:
wheeler(x, a=1.5) = -exp(-(x[1]*x[2] - a)^2 -(x[2]-a)^2) 

In [ ]:
x1=x2=range(0,3,50)
y = [wheeler([x_1,x_2]) for x_1 in x1, x_2 in x2]
plot(x1,x2,y)


In [ ]:
plot(x1,x2,y,st=:surface,camera=(-30,60))

### Exercise 1 — Explore the objective function

Use the code above to investigate Wheeler's ridge.

1. Create a contour plot in addition to the surface plot.
2. Mark the point $(1,1)$ on the contour plot.
3. Evaluate the function at:
   - $(1,1)$,
   - $(1.5,1.5)$,
   - $(2,1.5)$.
4. Change the parameter $a$ from `1.5` to `2.0`.
5. Describe how the location of the minimum changes.

Do not use an optimization routine yet.


In [ ]:
# Exercise 1

# Create a contour plot of Wheeler's ridge.
# Evaluate the function at the requested points.
# Repeat for a = 2.0.


## 2. Search directions and one-dimensional minimization

Several direct-search methods reduce a multidimensional problem to a sequence of one-dimensional searches.

For a current point $x$ and a direction $d$, define

$
\phi(\alpha)=f(x+\alpha d).
$

Instead of minimizing $f$ over all coordinates simultaneously, we minimize the scalar function $\phi(\alpha)$.

The helper function `basis(i,n)` creates the coordinate directions

$
e_1,e_2,\ldots,e_n.
$

The function `bracket_minimum` searches for an interval that contains a local minimum of a one-dimensional function.


In [ ]:
basis(i, n) = [k == i ? 1.0 : 0.0 for k in 1 : n] 

In [ ]:
function bracket_minimum(f, x=0; s=1e-2, k=2.0)
  a, ya = x, f(x)
  b, yb = a + s, f(a + s)
  if yb > ya
      a, b = b, a
      ya, yb = yb, ya
      s = -s
  end
  while true
      c, yc = b + s, f(b + s)
      if yc > yb
          return a < c ? (a, c) : (c, a)
      end
      a, ya, b, yb = b, yb, c, yc
      s *= k
  end
end

function line_search(f, x, d)
    objective = α -> f(x + α * d)
    a, b = bracket_minimum(objective)
  
    # Use Optim.jl to minimize the objective function
    result = optimize(objective, a, b, Brent())  # Brent's method for 1D minimization
    α = Optim.minimizer(result)  # Extract the minimizing α value
  
    return x + α * d
  end

### Exercise 2 — Coordinate directions

For $n=3$:

1. Generate `basis(1,3)`, `basis(2,3)`, and `basis(3,3)`.
2. Verify that the vectors are mutually orthogonal.
3. Let
   $
   x=(2,1)
   $
   and $d=e_1$.
   Define
   $
   \phi(\alpha)=f(x+\alpha d).
   $
4. Plot $\phi(\alpha)$ for $\alpha\in[-3,3]$.
5. Use `bracket_minimum` to find an interval containing a local minimum of $\phi$.


In [ ]:
# Exercise 2

# Generate coordinate directions.
# Define phi(alpha) = wheeler(x + alpha*d).
# Plot phi and bracket a minimum.


## 3. Cyclic coordinate search

Cyclic coordinate search optimizes one coordinate at a time.

Starting from $x^{(k)}$, the algorithm searches along

$
e_1,e_2,\ldots,e_n
$

in sequence. After the final coordinate has been processed, the procedure starts another cycle.

For each direction $e_i$, the method solves approximately

$
\min_{\alpha} f(x+\alpha e_i).
$

This approach is simple and requires no derivatives. However, it can be slow when the directions are poorly aligned with a narrow or curved valley.


In [ ]:
function cyclic_coordinate_descent(f, x, ϵ)
    Δ, n = Inf, length(x)
    while abs(Δ) > ϵ
        x′ = copy(x)
        for i in 1:n
            d = basis(i, n)
            x = line_search(f, x, d)
            @info d, x
        end
        Δ = norm(x - x′)
    end
    return x
end

In [ ]:
cyclic_coordinate_descent(wheeler,  [4.5,4.2], 10^(-5))

### Exercise 3 — Trace cyclic coordinate descent

Run cyclic coordinate descent on Wheeler's ridge using

```julia
x0 = [4.5, 4.2]
```

Modify the implementation so that it stores every accepted point.

Then:

1. plot the optimization trajectory on top of a contour plot;
2. count the number of iterations;
3. report the final point;
4. report the final objective value.

Repeat from

```julia
x0 = [0.5, 2.5]
```

and compare the two trajectories.


In [ ]:
# Exercise 3

# Modify or copy cyclic_coordinate_descent so that it stores its history.
# Plot the path on a contour plot.


### Exercise 4 — Effect of the stopping tolerance

Run cyclic coordinate descent with

$
\varepsilon \in \{10^{-2},10^{-3},10^{-4},10^{-5}\}.
$

For each tolerance, record:

- the final objective value;
- the number of iterations;
- the distance between the final point and the known optimum.

Summarize the results in a small table.

**Question:** What is gained, and what is the computational cost, when the tolerance becomes smaller?


In [ ]:
# Exercise 4

eps_values = [1e-2, 1e-3, 1e-4, 1e-5]

# Run the method for each tolerance and collect results.


## 4. Cyclic coordinate search with an acceleration step

A natural weakness of pure coordinate search is that it repeatedly moves only along the coordinate axes.

After completing a full cycle, we can use the net displacement

$
d = x_{\text{new}}-x_{\text{old}}
$

as an additional search direction.

This **acceleration step** may align better with the valley of the objective function and therefore reduce the number of cycles required.


In [ ]:
function cyclic_coordinate_descent_with_acceleration_step(f, x, ϵ)
    Δ, n = Inf, length(x)
    while abs(Δ) > ϵ
        x′ = copy(x)
        for i in 1:n
            d = basis(i, n)
            x = line_search(f, x, d)
            @info d,x
        end
        x = line_search(f, x, x - x′) # acceleration step
        @info "acceleration ",x
        Δ = norm(x - x′)
    end
    return x
end

In [ ]:
cyclic_coordinate_descent_with_acceleration_step(wheeler,  [4.5,4.2], 10^(-4))

### Exercise 5 — Does acceleration help?

Compare:

- standard cyclic coordinate descent;
- cyclic coordinate descent with an acceleration step.

Use the same:

- starting point;
- tolerance;
- objective function.

Record:

1. number of iterations;
2. final point;
3. final objective value;
4. total number of function evaluations, if you modify the code to count them.

Plot both trajectories on the same contour plot.

**Question:** Does the acceleration step always lead to a visibly shorter path?


In [ ]:
# Exercise 5

# Compare the two algorithms quantitatively and graphically.


## 5. Powell's method

Powell's method is another derivative-free direction-set method.

Instead of using only the fixed coordinate directions, Powell's method updates the set of directions during the search.

The basic idea is:

1. minimize successively along a collection of directions;
2. measure the total displacement over the cycle;
3. use this displacement as a new search direction;
4. replace one of the old directions.

This allows the direction set to adapt to the geometry of the objective function.

For quadratic functions, a good set of mutually conjugate directions can lead to very efficient optimization.


In [ ]:
function powell(f, x, ϵ)
    n = length(x)
    U = [basis(i, n) for i in 1:n]
    Δ = Inf
    while Δ > ϵ
        x′ = x
        for i in 1:n
            d = U[i]
            x′ = line_search(f, x′, d)
        end
        for i in 1:n-1
            U[i] = U[i+1]
        end
        U[n] = d = x′ - x
        x′ = line_search(f, x, d)
        Δ = norm(x′ - x)
        x = x′
        @info U,x
    end
    return x
end

In [ ]:
powell(wheeler,  [4.5,4.2], 10^(-4)) 

### Exercise 6 — Inspect Powell's directions

Modify the `powell` function so that it stores the direction set after each major iteration.

Then:

1. print the directions for the first three iterations;
2. examine how they differ from the original coordinate directions;
3. plot the optimization trajectory;
4. compare the trajectory with cyclic coordinate descent.

**Discussion:** Why can changing the search directions be advantageous?


In [ ]:
# Exercise 6

# Create a version of Powell's method that records the direction sets.


### Exercise 7 — Test Powell's method on another function

Apply Powell's method to the Rosenbrock function

$
f(x,y)=(1-x)^2+100(y-x^2)^2.
$

Use the starting point

$
x_0=(-1.2,1).
$

1. Plot the objective contours.
2. Run Powell's method.
3. Plot the optimization trajectory.
4. Report the final point and objective value.
5. Compare the behavior with Wheeler's ridge.

The Rosenbrock function is a useful test because it contains a long, narrow, curved valley.


In [ ]:
# Exercise 7

rosenbrock(x) = (1 - x[1])^2 + 100*(x[2] - x[1]^2)^2

# Run Powell's method and visualize the path.


## 6. Hooke–Jeeves pattern search

Hooke–Jeeves alternates between two ideas:

### Exploratory search

Starting from the current point, the method tests movements along the coordinate directions to find a better nearby point.

### Pattern move

If the exploratory search succeeds, the algorithm tries to continue in the same overall direction.

The initial step length is controlled by $\alpha$. When the exploratory search fails, the step length is reduced using

$
\alpha \leftarrow \gamma\alpha,
$

where typically $0<\gamma<1$.

The process continues until the step size becomes sufficiently small.


In [ ]:
function hooke_jeeves(f, x, α, ϵ, γ=0.5)
    y, n = f(x), length(x)
    while α > ϵ
        improved = false
        x_best, y_best = x, y
        for i in 1:n
            for sgn in (-1, 1)
                x′ = x + sgn * α * basis(i, n)
                y′ = f(x′)
                if y′ < y_best
                    x_best, y_best, improved = x′, y′, true
                end
            end
        end
        x, y = x_best, y_best
        if !improved
            α *= γ
        end
        @info x, y, α
    end
    return x
end

In [ ]:
x=hooke_jeeves(wheeler, [4.5,4.2], 3, 10^(-4)) 
x

### Exercise 8 — Effect of the initial step length

Run Hooke–Jeeves with

$
\alpha\in\{0.25,\;0.5,\;1,\;2,\;3\}
$

while keeping $\gamma=0.5$ and the same stopping tolerance.

For each value of $\alpha$, record:

- the final point;
- the final objective value;
- the number of iterations.

**Question:** How does a very small or very large initial step influence the search?


In [ ]:
# Exercise 8

alphas = [0.25, 0.5, 1.0, 2.0, 3.0]

# Compare Hooke-Jeeves for the different initial step lengths.


### Exercise 9 — Effect of the contraction factor

Fix the initial step length and compare

$
\gamma\in\{0.25,\;0.5,\;0.75,\;0.9\}.
$

Remember that $\gamma$ determines how quickly the exploratory step is reduced.

1. Which value gives the fastest reduction in step size?
2. Which value allows the finest gradual search?
3. How does $\gamma$ affect the number of iterations?


In [ ]:
# Exercise 9

gammas = [0.25, 0.5, 0.75, 0.9]

# Run Hooke-Jeeves for each value of gamma.


## 7. Comparing the direct-search methods

The algorithms studied in this notebook all avoid derivatives, but they use different strategies.

| Method | Main search directions | Adaptive directions? | Pattern/acceleration move? |
|---|---|---:|---:|
| Cyclic coordinate search | Coordinate axes | No | No |
| Accelerated cyclic search | Coordinate axes + displacement | Partly | Yes |
| Powell | Direction set | Yes | Yes |
| Hooke–Jeeves | Coordinate exploration | Indirectly | Yes |

A fair comparison should ideally use more than the final solution. Useful measures include:

- number of iterations;
- number of objective-function evaluations;
- final objective value;
- distance from the known optimum;
- robustness to the starting point.


### Exercise 10 — Mini benchmarking study

Compare all four methods on Wheeler's ridge using at least three starting points.

For each combination of method and starting point, record:

- final $x_1$;
- final $x_2$;
- final objective value;
- number of iterations;
- number of function evaluations.

Optional: present the results in a `DataFrame`.

Then create at least one graphical comparison, for example:

- function evaluations by method;
- final objective value by method;
- optimization trajectories.

### Discussion questions

1. Which method is most efficient for this problem?
2. Is the same method best from every starting point?
3. Which algorithm appears most sensitive to its parameters?
4. Why is the number of function evaluations often more informative than the number of iterations?


In [ ]:
# Exercise 10

# You may use:
# using DataFrames

# Build a small benchmarking experiment and store the results in a DataFrame.


## Summary

Direct-search methods are useful when derivative information is unavailable or undesirable.

In this notebook:

- **cyclic coordinate search** minimized along one coordinate at a time;
- an **acceleration step** used the displacement over a cycle as an additional direction;
- **Powell's method** adapted its direction set;
- **Hooke–Jeeves** combined exploratory and pattern moves.

The methods illustrate an important optimization principle: the choice of **search directions** can strongly influence efficiency, especially for narrow or curved valleys.

### Optional challenge

Choose a two-dimensional non-convex objective with several local minima.

Run all four direct-search methods from at least ten random starting points and investigate whether they converge to the same solution.

Visualize the final solutions on a contour plot.


In [ ]:
# Optional challenge
